# Chapter 41 — Agents, Tool Use, Cost, and Where They Break

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# An agent completes a task by taking several steps in sequence: plan,
# call a tool, read the result, decide the next step, repeat. If each
# step succeeds independently with probability p, the whole task
# succeeds only if every step does, and that compounds fast.
def simulate_task_success(n_steps, p_per_step, n_trials, seed):
    r = np.random.default_rng(seed)
    successes = r.random((n_trials, n_steps)) < p_per_step
    return successes.all(axis=1).mean()

print(f"{'steps':>7}{'p=0.99':>10}{'p=0.95':>10}{'p=0.90':>10}{'p=0.80':>10}")
for n_steps in (1, 3, 5, 10, 20, 40):
    row = [n_steps]
    for p in (0.99, 0.95, 0.90, 0.80):
        rate = simulate_task_success(n_steps, p, n_trials=20000, seed=41)
        row.append(rate)
    print(f"{row[0]:>7}{row[1]:>10.4f}{row[2]:>10.4f}{row[3]:>10.4f}{row[4]:>10.4f}")

print(f"\nexact formula: p^n_steps. at p=0.95, n=20: {0.95**20:.4f}")

### Block 2  (`c2.py`)

In [ ]:
# Retrying a failed step raises the effective per-step success rate, at
# a real cost: each retry is another full call to the model. Both the
# benefit and the cost need to be counted together.
def simulate_with_retries(n_steps, p_per_step, max_retries, n_trials, seed):
    r = np.random.default_rng(seed)
    total_calls = 0
    successes = 0
    for _ in range(n_trials):
        calls_this_trial = 0
        task_ok = True
        for _ in range(n_steps):
            attempt, ok = 0, False
            while attempt <= max_retries:
                calls_this_trial += 1
                attempt += 1
                if r.random() < p_per_step:
                    ok = True
                    break
            if not ok:
                task_ok = False
                break
        total_calls += calls_this_trial
        successes += task_ok
    return successes / n_trials, total_calls / n_trials

print(f"{'max retries':>12}{'task success rate':>19}{'avg calls per task':>20}")
for max_retries in (0, 1, 2, 3):
    rate, calls = simulate_with_retries(n_steps=10, p_per_step=0.90, max_retries=max_retries,
                                        n_trials=20000, seed=41)
    print(f"{max_retries:>12}{rate:>19.4f}{calls:>20.2f}")

print(f"\nat zero retries, average cost is BELOW ten calls, 6.54, because")
print(f"a task that fails partway through stops immediately rather than")
print(f"finishing all ten steps: failure is cheap here, precisely because")
print(f"nothing catches it. Each retry buys real accuracy at a real,")
print(f"compounding cost.")

### Block 3  (`c3.py`)

In [ ]:
# A transparent toy router: it picks a tool by keyword match against a
# task description. Simple enough to see exactly how it can fail, which
# is the point: real tool-routing failures in production agents look
# structurally like this, even when the router itself is a full model.
TOOLS = {
    "calculator": {"total", "average", "percent"},
    "search": {"who", "when", "where"},
    "code_exec": {"run", "debug"},
    "database": {"table", "customer"},
}

def route(words, confusion_prob, r):
    """With probability confusion_prob, add one keyword from a
    DIFFERENT tool into the task's own words, simulating a genuinely
    ambiguous or poorly-phrased sub-task description that legitimately
    points two ways at once."""
    words = set(words)
    if r.random() < confusion_prob:
        other_tool = r.choice([t for t in TOOLS if t not in
                               [tt for tt in TOOLS if words & TOOLS[tt]]] or list(TOOLS))
        words.add(r.choice(list(TOOLS[other_tool])))
    scores = {tool: len(words & kws) for tool, kws in TOOLS.items()}
    best = max(scores.values())
    tied = [t for t, s in scores.items() if s == best]
    return r.choice(tied)                              # ties broken at random, honestly

# each task has exactly ONE clean keyword: borderline by construction
tasks = [
    (["find", "the", "total", "for", "last", "month"], "calculator"),
    (["find", "who", "founded", "the", "company"], "search"),
    (["please", "run", "the", "validation", "job"], "code_exec"),
    (["look", "at", "the", "customer", "records"], "database"),
]

print(f"{'confusion rate':>15}{'per-call routing accuracy':>27}")
for confusion in (0.0, 0.15, 0.30, 0.50, 0.70):
    r = np.random.default_rng(41)
    correct, n_trials = 0, 4000
    for i in range(n_trials):
        words, true_tool = tasks[i % len(tasks)]
        picked = route(words, confusion, r)
        correct += (picked == true_tool)
    print(f"{confusion:>15.2f}{correct/n_trials:>27.4f}")

### Block 4  (`c4.py`)

In [ ]:
# Chain the router into a multi-step task: each step must pick the
# right tool AND that tool must succeed at its sub-task. A routing
# mistake is not usually recoverable mid-chain, since the wrong tool's
# output feeds directly into the next step.
def simulate_agent_task(n_steps, confusion, tool_success_given_correct, n_trials, seed):
    r = np.random.default_rng(seed)
    successes = 0
    for _ in range(n_trials):
        ok = True
        for step in range(n_steps):
            words, true_tool = tasks[step % len(tasks)]
            picked = route(words, confusion, r)
            if picked != true_tool:
                ok = False
                break
            if r.random() >= tool_success_given_correct:
                ok = False
                break
        successes += ok
    return successes / n_trials

print(f"{'steps':>7}{'confusion=0.0':>15}{'confusion=0.15':>16}{'confusion=0.30':>16}")
for n_steps in (1, 3, 5, 10):
    row = [n_steps]
    for confusion in (0.0, 0.15, 0.30):
        rate = simulate_agent_task(n_steps, confusion, tool_success_given_correct=0.95,
                                   n_trials=8000, seed=41)
        row.append(rate)
    print(f"{row[0]:>7}{row[1]:>15.4f}{row[2]:>16.4f}{row[3]:>16.4f}")

print(f"\neven a well-behaved 95%-reliable tool cannot rescue a chain that")
print(f"keeps routing to the wrong tool in the first place.")

### Block 5  (`c5.py`)

In [ ]:
# An agent must also decide WHEN to stop: continuing after the real
# answer is found wastes cost, and stopping too early returns an
# incomplete result. A confidence threshold controls this tradeoff
# directly, and the right setting depends on how those two costs compare.
def simulate_stopping(threshold, true_step_needed, max_steps, n_trials, seed):
    r = np.random.default_rng(seed)
    premature, wasted_steps, correct_stops = 0, 0, 0
    for _ in range(n_trials):
        stopped_at = None
        for step in range(1, max_steps + 1):
            # confidence rises noisily as the agent approaches the true answer
            true_progress = min(step / true_step_needed, 1.0)
            confidence = np.clip(true_progress + r.normal(0, 0.12), 0, 1)
            if confidence >= threshold:
                stopped_at = step
                break
        if stopped_at is None:
            stopped_at = max_steps
        if stopped_at < true_step_needed:
            premature += 1
        else:
            wasted_steps += (stopped_at - true_step_needed)
            correct_stops += 1
    return premature / n_trials, wasted_steps / n_trials

print(f"{'threshold':>11}{'premature-stop rate':>21}{'avg wasted steps':>18}")
for threshold in (0.5, 0.7, 0.85, 0.95, 0.99):
    premature, wasted = simulate_stopping(threshold, true_step_needed=8, max_steps=20,
                                          n_trials=10000, seed=41)
    print(f"{threshold:>11.2f}{premature:>21.4f}{wasted:>18.3f}")

print(f"\na low threshold stops early, often before the task is actually done;")
print(f"a high threshold rarely stops early, at the cost of extra steps")
print(f"spent confirming what was already true.")